In [1]:
from pathlib import Path
import lsstypes as types
import matplotlib.pyplot as plt

In [ ]:
dirname = Path('/pscratch/sd/a/adematti/measurements/angular_shotnoise')

region = 'SGC'
stats = ['mesh2_spectrum', 'mesh3_spectrum']
imweights = ['WEIGHT_SYS', 'WEIGHT_IMLIN', 'WEIGHT_IMLIN_DES']
randomizes = ['isotropic', 'angular', 'isotropic_norm', 'angular_norm']
norm_regions_list = ['NS', 'NSnoDESDES']

spectra = {}
for stat in stats:
    for norm_regions in norm_regions_list:
        for imweight in imweights:
            for randomize in [None] + randomizes:
                if randomize is not None and imweight != 'WEIGHT_IMLIN_DES':
                    continue  # randomize shown for WEIGHT_IMLIN_DES only
                randomize_label = f'_randomize-{randomize}' if randomize else ''
                fn = dirname / f'{stat}_ELG_LOPnotqso_{region}_{imweight}_norm{norm_regions}{randomize_label}.h5'
                spectrum = types.read(fn)
                if stat == 'mesh2_spectrum':
                    spectrum = spectrum.select(k=slice(0, None, 5))
                spectra[stat, norm_regions, imweight, randomize] = spectrum

In [ ]:
for norm_regions in norm_regions_list:
    ells = spectra['mesh2_spectrum', norm_regions, 'WEIGHT_SYS', None].ells[:2]
    fig, lax = plt.subplots(len(ells), sharex=True, figsize=(7., 6.))
    lax = lax.ravel()
    for ill, ell in enumerate(ells):
        ax = lax[ill]
        ref = spectra['mesh2_spectrum', norm_regions, 'WEIGHT_SYS', None].get(ell)
        k = ref.coords('k')
        ax.plot(k, k * (spectra['mesh2_spectrum', norm_regions, 'WEIGHT_IMLIN', None].get(ell).value() - ref.value()), color='C0', label='WEIGHT_IMLIN')
        ax.plot(k, k * (spectra['mesh2_spectrum', norm_regions, 'WEIGHT_IMLIN_DES', None].get(ell).value() - ref.value()), color='C1', label='WEIGHT_IMLIN_DES')
        for irandomize, randomize in enumerate(randomizes):
            ax.plot(k, k * (spectra['mesh2_spectrum', norm_regions, 'WEIGHT_IMLIN_DES', randomize].get(ell).value() - ref.value()),
                    color=f'C{2 + irandomize:d}', linestyle='--', label=f'WEIGHT_IMLIN_DES {randomize}')
        ax.set_ylabel(rf'$k \Delta P_{ell:d}(k)$ [$(\mathrm{{Mpc}}/h)^2$]')
        ax.set_ylim(-1., 10.)
        ax.grid(True)
    lax[0].legend(fontsize='small', ncols=2)
    lax[0].set_title(f'{region}, norm regions: {norm_regions}')
    lax[-1].set_xlabel(r'$k$ [$h/\mathrm{Mpc}$]')
    plt.show()

In [ ]:
for norm_regions in norm_regions_list:
    ells = spectra['mesh3_spectrum', norm_regions, 'WEIGHT_SYS', None].ells
    fig, lax = plt.subplots(len(ells), sharex=True, figsize=(7., 6.))
    lax = lax.ravel()
    for ill, ell in enumerate(ells):
        ax = lax[ill]
        ref = spectra['mesh3_spectrum', norm_regions, 'WEIGHT_SYS', None].get(ell)
        k = ref.coords('k')[..., 0]
        ax.plot(k, k**2 * (spectra['mesh3_spectrum', norm_regions, 'WEIGHT_IMLIN', None].get(ell).value() - ref.value()), color='C0', label='WEIGHT_IMLIN')
        ax.plot(k, k**2 * (spectra['mesh3_spectrum', norm_regions, 'WEIGHT_IMLIN_DES', None].get(ell).value() - ref.value()), color='C1', label='WEIGHT_IMLIN_DES')
        for irandomize, randomize in enumerate(randomizes):
            ax.plot(k, k**2 * (spectra['mesh3_spectrum', norm_regions, 'WEIGHT_IMLIN_DES', randomize].get(ell).value() - ref.value()),
                    color=f'C{2 + irandomize:d}', linestyle='--', label=f'WEIGHT_IMLIN_DES {randomize}')
        ax.set_ylabel(rf'$k^2 \Delta B_{{{ell[0]:d}{ell[1]:d}{ell[2]:d}}}(k, k)$ [$(\mathrm{{Mpc}}/h)^4$]')
        ax.grid(True)
    lax[0].legend(fontsize='small', ncols=2)
    lax[0].set_title(f'{region}, norm regions: {norm_regions}')
    lax[-1].set_xlabel(r'$k$ [$h/\mathrm{Mpc}$]')
    plt.show()